In [1]:
import os
import json
import glob
import pandas as pd
from collections import defaultdict
from pathlib import Path

In [2]:
# input
folder = "16APR2026"

In [3]:
pwd

'D:\\Study\\Programs\\trading'

In [4]:
# =========================
# 1. Configuration
# =========================

log_folder = Path(f"assets/logs/{folder}/ticks")
output_folder = Path(f"assets/logs/{folder}/extracted_symbols")

os.makedirs(output_folder, exist_ok=True)

log_pattern = os.path.join(log_folder, f"{folder}_ticks.log*")
log_files = glob.glob(log_pattern)

print(f"Found {len(log_files)} log files.")
if not log_files:
    print("⚠️ No files found. Check folder name or pattern.")
    exit()

# =========================
# 2. Processing with Batching
# =========================
lines_processed = 0
lines_matched = 0
errors = 0
unique_symbols = set()

# Configuration for batching
BATCH_LIMIT = 50000  # Dump to disk every 50,000 matched lines
current_batch_size = 0
buffers = defaultdict(list)

def flush_buffers():
    """Writes all buffered data to disk and clears memory to avoid OS file limits."""
    for sym, lines in buffers.items():
        output_path = os.path.join(output_folder, f"{sym}.json")
        with open(output_path, 'a', encoding='utf-8') as out_file:
            # Join with newline and ensure a trailing newline
            out_file.write('\n'.join(lines) + '\n')
    buffers.clear()

try:
    for file_path in log_files:
        print(f"Processing: {file_path}")

        with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
            for line in file:
                lines_processed += 1

                if "INFO - tick_data:" not in line:
                    continue

                lines_matched += 1

                try:
                    parts = line.split(" | ", 1)
                    if len(parts) < 2:
                        continue

                    json_string = parts[1].strip()

                    # Parse JSON to get symbol
                    tick_data = json.loads(json_string)
                    symbol = tick_data.get("symbol")

                    if not symbol:
                        continue

                    unique_symbols.add(symbol)

                    # Add to buffer instead of writing directly
                    buffers[symbol].append(json_string)
                    current_batch_size += 1

                    # Flush if batch limit is reached
                    if current_batch_size >= BATCH_LIMIT:
                        flush_buffers()
                        current_batch_size = 0

                except json.JSONDecodeError as e:
                    errors += 1
                    print(f"JSON error in {file_path}: {e}")

                except Exception as e:
                    errors += 1
                    print(f"Unexpected error in {file_path}: {e}")

finally:
    # =========================
    # 3. Cleanup: Flush remaining data
    # =========================
    if current_batch_size > 0:
        flush_buffers()

# =========================
# 4. Sorting the Output Files
# =========================
print("\nSorting extracted files by timestamp...")

for sym in unique_symbols:
    file_path = os.path.join(output_folder, f"{sym}.json")

    if os.path.exists(file_path):
        try:
            # Read all JSON lines into a list of dictionaries
            with open(file_path, 'r', encoding='utf-8') as f:
                ticks = [json.loads(line) for line in f if line.strip()]

            # Sort the list.
            # Key logic: Use last_trade_time if it exists, otherwise use exchange_timestamp
            if sym == "NIFTY 50":
                ticks.sort(key=lambda x: x.get("local_time"))
            else:
                ticks.sort(key=lambda x: x.get("last_trade_time"))

            # Overwrite the file with the sorted data
            with open(file_path, 'w', encoding='utf-8') as f:
                for tick in ticks:
                    f.write(json.dumps(tick) + '\n')

        except Exception as e:
            print(f"Error sorting file {file_path}: {e}")

# =========================
# 5. Summary
# =========================
print("\n===== SUMMARY =====")
print(f"Total lines processed : {lines_processed}")
print(f"Matching lines        : {lines_matched}")
print(f"Errors                : {errors}")
print(f"Symbols extracted     : {len(unique_symbols)}")
print(f"Output folder         : {output_folder}")
print("Extraction and sorting complete.")

Found 4 log files.
Processing: assets\logs\16APR2026\ticks\16APR2026_ticks.log
Processing: assets\logs\16APR2026\ticks\16APR2026_ticks.log.1
Processing: assets\logs\16APR2026\ticks\16APR2026_ticks.log.2
Processing: assets\logs\16APR2026\ticks\16APR2026_ticks.log.3

Sorting extracted files by timestamp...

===== SUMMARY =====
Total lines processed : 179311
Matching lines        : 175479
Errors                : 0
Symbols extracted     : 27
Output folder         : assets\logs\16APR2026\extracted_symbols
Extraction and sorting complete.


In [5]:
import os
import glob
import pandas as pd

def convert_json_to_excel(folder_path, is_json_lines=True):
    """
    Reads all JSON files in a folder and saves them as Excel files.

    :param folder_path: The directory containing the .json files.
    :param is_json_lines: Set to True if the JSON file has one JSON object per line.
    """

    # =========================
    # Define desired column order
    # =========================
    desired_order = [
        "instrument_token", "symbol", "exchange_timestamp", "local_time",
        "last_trade_time", "last_price", "last_traded_quantity",
        "average_traded_price", "option_CE_PE", "option_type", "strike",
        "volume_traded", "total_buy_quantity", "total_sell_quantity",
        "ohlc", "change", "oi", "oi_day_high", "oi_day_low", "depth"
    ]

    # Create a search pattern to find all .json files in the folder
    search_pattern = os.path.join(folder_path, "*.json")
    json_files = glob.glob(search_pattern)

    if not json_files:
        print(f"No JSON files found in {folder_path}")
        return

    print(f"Found {len(json_files)} JSON files. Starting conversion...\n")

    successful_conversions = 0
    errors = 0

    for json_file in json_files:
        try:
            # 1. Load the JSON file into a Pandas DataFrame
            df = pd.read_json(json_file, lines=is_json_lines)

            # =========================
            # Reorder columns safely
            # =========================
            # Keep only the columns from desired_order that actually exist in this specific DataFrame
            ordered_cols = [col for col in desired_order if col in df.columns]

            # Find any extra columns present in the DataFrame that are NOT in desired_order
            extra_cols = [col for col in df.columns if col not in desired_order]

            # Apply the new column order (desired ones first, any unexpected extras at the end)
            df = df[ordered_cols + extra_cols]


            # 2. Construct the new Excel file path
            base_name = os.path.splitext(json_file)[0]
            excel_file = f"{base_name}.xlsx"

            # 3. Dump the DataFrame to Excel
            df.to_excel(excel_file, index=False, engine='openpyxl')

            print(f"Success: {os.path.basename(json_file)} -> {os.path.basename(excel_file)}")
            successful_conversions += 1

        except ValueError as ve:
            print(f"Format Error in {os.path.basename(json_file)}: {ve}")
            print("Tip: If it's a standard JSON file, try setting is_json_lines=False")
            errors += 1
        except Exception as e:
            print(f"Error processing {os.path.basename(json_file)}: {e}")
            errors += 1

    print("\n===== SUMMARY =====")
    print(f"Total files found : {len(json_files)}")
    print(f"Successfully saved: {successful_conversions}")
    print(f"Errors            : {errors}")


# Specify the path to your extracted symbols folder
# Example: "logs/02APR2026/extracted_symbols"
target_folder = Path(f"assets/logs/{folder}/extracted_symbols")

convert_json_to_excel(output_folder, is_json_lines=True)

Found 27 JSON files. Starting conversion...

Success: NIFTY 50.json -> NIFTY 50.xlsx
Success: NIFTY26JUN23950CE.json -> NIFTY26JUN23950CE.xlsx
Success: NIFTY26JUN23950PE.json -> NIFTY26JUN23950PE.xlsx
Success: NIFTY26JUN24050CE.json -> NIFTY26JUN24050CE.xlsx
Success: NIFTY26JUN24050PE.json -> NIFTY26JUN24050PE.xlsx
Success: NIFTY26JUN24100CE.json -> NIFTY26JUN24100CE.xlsx
Success: NIFTY26JUN24100PE.json -> NIFTY26JUN24100PE.xlsx
Success: NIFTY26JUN24150CE.json -> NIFTY26JUN24150CE.xlsx
Success: NIFTY26JUN24150PE.json -> NIFTY26JUN24150PE.xlsx
Success: NIFTY26JUN24200CE.json -> NIFTY26JUN24200CE.xlsx
Success: NIFTY26JUN24200PE.json -> NIFTY26JUN24200PE.xlsx
Success: NIFTY26JUN24250CE.json -> NIFTY26JUN24250CE.xlsx
Success: NIFTY26JUN24250PE.json -> NIFTY26JUN24250PE.xlsx
Success: NIFTY26JUN24300CE.json -> NIFTY26JUN24300CE.xlsx
Success: NIFTY26JUN24300PE.json -> NIFTY26JUN24300PE.xlsx
Success: NIFTY26JUN24350CE.json -> NIFTY26JUN24350CE.xlsx
Success: NIFTY26JUN24350PE.json -> NIFTY26JUN

In [9]:
Path(f"assets/logs/{folder}/extracted_symbols")

WindowsPath('assets/logs/16APR2026/extracted_symbols')

In [8]:
pwd

'D:\\Study\\Programs\\trading'